# B3 v6 — Devign validation; REVEAL blocked
CPU-prepared delivery. Native GPU acceptance is required before claiming runtime PASS.
Account: huuhieu3333. Set explicit checkpoint path and RESUME_REQUIRED=True for resume.


In [ ]:
from pathlib import Path
import sys, os, json, subprocess, traceback, shutil, platform, importlib.metadata, zipfile, hashlib, time, re
from datetime import datetime, timezone

SESSION_STARTED = time.time()
DIAG = Path('/kaggle/working/grace_diagnostics')
DIAG.mkdir(parents=True, exist_ok=True)
EVENTS = DIAG / 'events.log'

def note(message):
    text = str(message)
    print(text, flush=True)
    with EVENTS.open('a', encoding='utf-8') as f:
        f.write(f'[{datetime.now(timezone.utc).isoformat()}] {text}\n')

def diagnostics_zip():
    archive = shutil.make_archive('/kaggle/working/grace_diagnostics', 'zip', root_dir=DIAG)
    print('Tải log chẩn đoán từ Kaggle Output:', archive, flush=True)
    return archive

def run_logged(cmd, name, env=None):
    log = DIAG / f'{name}.log'
    note(f'Bắt đầu {name}: {cmd!r} | log đầy đủ: {log}')
    is_inference = name.startswith('inference_seed_')
    with log.open('w', encoding='utf-8') as f:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, encoding='utf-8', errors='replace', bufsize=1, env=env)
        for line in proc.stdout:
            f.write(line); f.flush()
            if re.search(r'\b(?:Found|Skip(?:ping)?) link\b', line, flags=re.IGNORECASE):
                continue
            if is_inference:
                match = re.search(r'^(\S+).*\((\d+)/(\d+)\)\s*$', line.strip())
                if match:
                    run_key, done, total = match.group(1), int(match.group(2)), int(match.group(3))
                    if done % 100 == 0 or done == total:
                        print(f'Inference {run_key}: {done}/{total}', flush=True)
                elif line.strip():
                    print(line, end='', flush=True)
            else:
                print(line, end='', flush=True)
        code = proc.wait()
    if is_inference and '--status' in cmd:
        status_path = Path(cmd[cmd.index('--status') + 1])
        if status_path.is_file():
            status = json.loads(status_path.read_text(encoding='utf-8'))
            print(f"Inference summary: {status.get('completed_records_total', 0)} records; state={status.get('state')}", flush=True)
            for run_key, progress in status.get('runs', {}).items():
                print(f"  {run_key}: {progress.get('completed', 0)}/{progress.get('count', '?')} [{progress.get('state', 'pending')}]")
    note(f'{name} kết thúc với mã {code}')
    if code:
        raise subprocess.CalledProcessError(code, cmd)

def versions():
    result = {}
    for name in ('llama-cpp-python', 'transformers', 'huggingface_hub', 'scikit-learn', 'numpy', 'scipy', 'tokenizers'):
        try: result[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError: result[name] = 'not installed'
    note('Installed package versions: ' + json.dumps(result))
    return result

def capture_failure(shell, etype, evalue, tb, tb_offset=None):
    trace = ''.join(traceback.format_exception(etype, evalue, tb))
    try:
        with (DIAG / 'exceptions.log').open('a', encoding='utf-8') as f:
            f.write(f'\n[{datetime.now(timezone.utc).isoformat()}]\n{trace}\n')
        note(f'Lỗi: {etype.__name__}: {evalue}')
        versions()
        diagnostics_zip()
    except Exception as diagnostic_error:
        print('Không thể ghi đầy đủ chẩn đoán:', diagnostic_error, flush=True)
    shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)
    return []

get_ipython().set_custom_exc((Exception,), capture_failure)
note('UTC: ' + datetime.now(timezone.utc).isoformat())
note('Python: ' + sys.version)
note('Executable: ' + sys.executable)
note('Platform: ' + platform.platform())
for loc in ('/kaggle/working', '/kaggle/input'):
    try:
        usage = shutil.disk_usage(loc)
        note(f'Disk {loc}: total={usage.total} free={usage.free} bytes')
    except OSError as exc: note(f'Disk {loc}: {exc}')
versions()
for name, cmd in [('gpu', ['nvidia-smi']), ('cuda_compiler', ['nvcc', '--version']), ('cmake', ['cmake', '--version']), ('pip', [sys.executable, '-m', 'pip', '--version'])]:
    try: run_logged(cmd, name)
    except (OSError, subprocess.CalledProcessError) as exc: note(f'{name}: {type(exc).__name__}: {exc}')
try:
    input_entries = [p.name for p in Path('/kaggle/input').glob('*')]
    note(f'Input entries: {input_entries}')
    gguf_files = [str(p) for p in Path('/kaggle/input').rglob('*.gguf')]
    note(f'Input GGUF files: {gguf_files}')
except Exception as exc: note(f'Input inspection error: {exc}')
diagnostics_zip()


In [ ]:
from pathlib import Path
import base64, hashlib, os, sys
raw = base64.b64decode('IiIiRmFpbC1jbG9zZWQgQjMgZGVsaXZlcnkgSU8uIE5vIG5ldHdvcmssIGJhY2tlbmQgb3IgZnJvemVuLWRhdGEgbXV0YXRpb25zLiIiIgppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBvcnQgbWF0aAppbXBvcnQgb3MKaW1wb3J0IHJlCmltcG9ydCBzaHV0aWwKaW1wb3J0IHRlbXBmaWxlCmltcG9ydCB6aXBmaWxlCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aCwgUHVyZVBvc2l4UGF0aAoKCmRlZiBzaGEocmF3KToKICAgIHJldHVybiBoYXNobGliLnNoYTI1NihyYXcpLmhleGRpZ2VzdCgpCgoKZGVmIGVuY29kZWQob2JqKToKICAgIHJldHVybiAoanNvbi5kdW1wcyhvYmosIHNvcnRfa2V5cz1UcnVlLCBlbnN1cmVfYXNjaWk9RmFsc2UsIGluZGVudD0yKSArICdcbicpLmVuY29kZSgndXRmLTgnKQoKCmRlZiBzZWFsKG9iaiwga2V5PSdzZWFsX3NoYTI1NicpOgogICAgYm9keSA9IHtrOiB2IGZvciBrLCB2IGluIG9iai5pdGVtcygpIGlmIGsgIT0ga2V5fQogICAgcmV0dXJuIHNoYShqc29uLmR1bXBzKGJvZHksIHNvcnRfa2V5cz1UcnVlLCBlbnN1cmVfYXNjaWk9RmFsc2UsIGluZGVudD0yKS5lbmNvZGUoJ3V0Zi04JykpCgoKZGVmIGNoZWNrZWRfbmFtZShuYW1lKToKICAgIGlmIG5vdCBuYW1lIG9yICdcXCcgaW4gbmFtZSBvciAnOicgaW4gbmFtZSBvciAnXHgwMCcgaW4gbmFtZToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdVbnNhZmUgbWVtYmVyIHBhdGgnKQogICAgcCA9IFB1cmVQb3NpeFBhdGgobmFtZSkKICAgIGlmIHAuaXNfYWJzb2x1dGUoKSBvciBhbnkocyBpbiAoJycsICcuJywgJy4uJykgZm9yIHMgaW4gbmFtZS5zcGxpdCgnLycpKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdVbnNhZmUgbWVtYmVyIHBhdGgnKQogICAgcmV0dXJuIG5hbWUKCgpkZWYgcmVhZF9tZW1iZXJzKHNvdXJjZSk6CiAgICBzb3VyY2UgPSBQYXRoKHNvdXJjZSkKICAgIGlmIHNvdXJjZS5pc19maWxlKCk6CiAgICAgICAgd2l0aCB6aXBmaWxlLlppcEZpbGUoc291cmNlKSBhcyB6OgogICAgICAgICAgICBuYW1lcyA9IHoubmFtZWxpc3QoKQogICAgICAgICAgICBpZiBsZW4obmFtZXMpICE9IGxlbihzZXQobmFtZXMpKToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0R1cGxpY2F0ZSBhcmNoaXZlIG1lbWJlcicpCiAgICAgICAgICAgIHJlc3VsdCA9IHt9CiAgICAgICAgICAgIGZvciBpbmZvIGluIHouaW5mb2xpc3QoKToKICAgICAgICAgICAgICAgIGlmIGluZm8uaXNfZGlyKCk6CiAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIG5hbWUgPSBjaGVja2VkX25hbWUoaW5mby5maWxlbmFtZSkKICAgICAgICAgICAgICAgIGlmIChpbmZvLmV4dGVybmFsX2F0dHIgPj4gMTYpICYgMG8xNzAwMDAgPT0gMG8xMjAwMDA6CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignU3ltbGluayBhcmNoaXZlIG1lbWJlcicpCiAgICAgICAgICAgICAgICByZXN1bHRbbmFtZV0gPSB6LnJlYWQoaW5mbykKICAgICAgICAgICAgcmV0dXJuIHJlc3VsdAogICAgaWYgbm90IHNvdXJjZS5pc19kaXIoKSBvciBzb3VyY2UuaXNfc3ltbGluaygpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ01pc3NpbmcgaW5wdXQgcm9vdCcpCiAgICByZXN1bHQgPSB7fQogICAgZm9yIHAgaW4gc291cmNlLnJnbG9iKCcqJyk6CiAgICAgICAgaWYgcC5pc19zeW1saW5rKCk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ1N5bWxpbmsgaW5wdXQgbWVtYmVyJykKICAgICAgICBpZiBwLmlzX2ZpbGUoKToKICAgICAgICAgICAgcmVzdWx0W2NoZWNrZWRfbmFtZShwLnJlbGF0aXZlX3RvKHNvdXJjZSkuYXNfcG9zaXgoKSldID0gcC5yZWFkX2J5dGVzKCkKICAgIHJldHVybiByZXN1bHQKCgpkZWYgdmVyaWZ5X2ludmVudG9yeShtZW1iZXJzLCBleHBlY3RlZCk6CiAgICBpZiBzZXQobWVtYmVycykgIT0gc2V0KGV4cGVjdGVkKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdNaXNzaW5nL3VuZXhwZWN0ZWQgbWVtYmVycycpCiAgICBmb3IgbmFtZSwgaCBpbiBleHBlY3RlZC5pdGVtcygpOgogICAgICAgIGNoZWNrZWRfbmFtZShuYW1lKQogICAgICAgIGlmIHNoYShtZW1iZXJzW25hbWVdKSAhPSBoOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdNZW1iZXIgU0hBLTI1NiBtaXNtYXRjaDogJyArIG5hbWUpCgoKZGVmIGxvYWRfaW5wdXRfcm9vdChwYXRoLCBleHBlY3RlZF9zaGEpOgogICAgcmF3ID0gUGF0aChwYXRoKS5yZWFkX2J5dGVzKCkKICAgIGlmIHNoYShyYXcpICE9IGV4cGVjdGVkX3NoYToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdJbmRlcGVuZGVudCBpbnB1dCByb290IHBpbiBtaXNtYXRjaCcpCiAgICByb290ID0ganNvbi5sb2FkcyhyYXcpCiAgICBpZiByb290LmdldCgnc2VhbF9zaGEyNTYnKSAhPSBzZWFsKHJvb3QpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0lucHV0IHJvb3Qgc2VhbCBtaXNtYXRjaCcpCiAgICBpZiByb290WydzY29wZSddWydhcm0nXSAhPSAnQjMnIG9yIHJvb3RbJ3Njb3BlJ11bJ3NwbGl0J10gIT0gJ3ZhbGlkYXRpb24nOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ09ubHkgQjMgdmFsaWRhdGlvbiBzdXBwb3J0ZWQgYnkgdGhpcyBkZWxpdmVyeScpCiAgICByZXR1cm4gcm9vdAoKCmRlZiB2YWxpZGF0ZV9idW5kbGUoc291cmNlLCByb290KToKICAgIGlmIFBhdGgoc291cmNlKS5pc19maWxlKCkgYW5kIHNoYShQYXRoKHNvdXJjZSkucmVhZF9ieXRlcygpKSAhPSByb290WydidW5kbGVfc2hhMjU2J106CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignRXh0ZXJuYWwgYnVuZGxlIHBpbiBtaXNtYXRjaCcpCiAgICBtZW1iZXJzID0gcmVhZF9tZW1iZXJzKHNvdXJjZSkKICAgIHZlcmlmeV9pbnZlbnRvcnkobWVtYmVycywgcm9vdFsnbWVtYmVyc19zaGEyNTYnXSkKICAgIGJtID0ganNvbi5sb2FkcyhtZW1iZXJzWydidW5kbGVfbWFuaWZlc3QuanNvbiddKQogICAgaWYgYm0uZ2V0KCdtYW5pZmVzdF9zZWFsX3NoYTI1NicpICE9IHNlYWwoYm0sICdtYW5pZmVzdF9zZWFsX3NoYTI1NicpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0J1bmRsZSBzZWFsIG1pc21hdGNoJykKICAgIGlmIGJtWydzY29wZSddICE9IHJvb3RbJ3Njb3BlJ10gb3IgYm1bJ21ldGhvZF9pZCddICE9IHJvb3RbJ21ldGhvZF9pZCddOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0J1bmRsZSBzY29wZS9tZXRob2QgbWlzbWF0Y2gnKQogICAgcm93cyA9IFtqc29uLmxvYWRzKGwpIGZvciBsIGluIG1lbWJlcnNbJ3Byb21wdHMuanNvbmwnXS5kZWNvZGUoJ3V0Zi04Jykuc3BsaXRsaW5lcygpIGlmIGwuc3RyaXAoKV0KICAgIGlmIGxlbihyb3dzKSAhPSByb290WydzY29wZSddWydjb3VudCddIG9yIGxlbih7clsnaWQnXSBmb3IgciBpbiByb3dzfSkgIT0gbGVuKHJvd3MpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0NlbnN1cyBjb3VudC9JRCBtaXNtYXRjaCcpCiAgICBmb3Igcm93IGluIHJvd3M6CiAgICAgICAgaWYgKG5vdCBpc2luc3RhbmNlKHJvd1snaWQnXSwgc3RyKSBvciByb3dbJ2xhYmVsJ10gbm90IGluICgwLCAxKQogICAgICAgICAgICAgICAgb3IgdHlwZShyb3dbJ2xhYmVsJ10pIGlzIG5vdCBpbnQgb3Igbm90IHJvd1snaWQnXS5zdGFydHN3aXRoKHJvb3RbJ3Njb3BlJ11bJ2RhdGFzZXQnXSsnOnZhbGlkYXRpb246JykKICAgICAgICAgICAgICAgIG9yIChyb3dbJ2RhdGFzZXQnXSwgcm93WydzcGxpdCddKSAhPSAocm9vdFsnc2NvcGUnXVsnZGF0YXNldCddLCAndmFsaWRhdGlvbicpCiAgICAgICAgICAgICAgICBvciBzaGEocm93Wydwcm9tcHQnXS5lbmNvZGUoJ3V0Zi04JykpICE9IHJvd1sncHJvbXB0X3NoYTI1NiddKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignSW52YWxpZCBpbmZlcmVuY2Ugcm93JykKICAgIHJldHVybiBibSwgcm93cywgbWVtYmVycwoKCmRlZiBkaXNjb3Zlcl9idW5kbGUoaW5wdXRfZGlyLCByb290KToKICAgIGlucHV0X2RpciA9IFBhdGgoaW5wdXRfZGlyKQogICAgcGFja2VkID0gc29ydGVkKGlucHV0X2Rpci5yZ2xvYihyb290WydidW5kbGVfZmlsZW5hbWUnXSkpCiAgICB1bnBhY2tlZCA9IHNvcnRlZChwLnBhcmVudCBmb3IgcCBpbiBpbnB1dF9kaXIucmdsb2IoJ2J1bmRsZV9tYW5pZmVzdC5qc29uJykpCiAgICBjYW5kaWRhdGVzID0gcGFja2VkICsgdW5wYWNrZWQKICAgIGlmIGxlbihjYW5kaWRhdGVzKSAhPSAxOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0V4cGVjdGVkIGV4YWN0bHkgb25lIHBhY2tlZCBvciB1bnBhY2tlZCBpbnB1dCByb290JykKICAgIHZhbGlkYXRlX2J1bmRsZShjYW5kaWRhdGVzWzBdLCByb290KQogICAgcmV0dXJuIGNhbmRpZGF0ZXNbMF0KCgpkZWYgd3JpdGVfemlwKHBhdGgsIG1lbWJlcnMpOgogICAgcGF0aCA9IFBhdGgocGF0aCkKICAgIHBhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHdpdGggemlwZmlsZS5aaXBGaWxlKHBhdGgsICd3JywgY29tcHJlc3Npb249emlwZmlsZS5aSVBfREVGTEFURUQpIGFzIHo6CiAgICAgICAgZm9yIG5hbWUsIHJhdyBpbiBzb3J0ZWQobWVtYmVycy5pdGVtcygpKToKICAgICAgICAgICAgaW5mbyA9IHppcGZpbGUuWmlwSW5mbyhjaGVja2VkX25hbWUobmFtZSksIGRhdGVfdGltZT0oMjAyNiwgMTEsIDgsIDAsIDAsIDApKQogICAgICAgICAgICBpbmZvLmNvbXByZXNzX3R5cGUgPSB6aXBmaWxlLlpJUF9ERUZMQVRFRAogICAgICAgICAgICB6LndyaXRlc3RyKGluZm8sIHJhdykKCgpkZWYgcnVuX2lkZW50aXR5KHJvb3QpOgogICAgcmV0dXJuIHtrOiByb290W2tdIGZvciBrIGluICgnbWV0aG9kX2lkJywgJ3JlbGVhc2VfaWQnLCAnc2NvcGUnLCAnbW9kZWxfc2hhMjU2JywKICAgICAgICAncnVudGltZV9jb250cmFjdF9zaGEyNTYnLCAndG9rZW5pemVyX3JldmlzaW9uJywgJ3Bvb2xfc2hhMjU2JywgJ3Byb21wdF9jb250ZW50X3NoYTI1NicsCiAgICAgICAgJ21lbWJlcnNfc2hhMjU2JywgJ3Njb3JpbmcnLCAnY29kZV9zaGEyNTYnKX0KCgpkZWYgY2hlY2tlZF9yZWNvcmRzKG91dF9yb290LCByb290LCByb3dzKToKICAgIG91dF9yb290ID0gUGF0aChvdXRfcm9vdCkKICAgIHNjb3BlID0gcm9vdFsnc2NvcGUnXQogICAgZm9sZGVyID0gb3V0X3Jvb3QgLyAoJ3NlZWRfJytzdHIoc2NvcGVbJ3NlZWQnXSkpIC8gc2NvcGVbJ2RhdGFzZXQnXSAvIHNjb3BlWydzcGxpdCddCiAgICByZWNvcmRzID0gZm9sZGVyLydyZWNvcmRzLmpzb25sJwogICAgaWYgbm90IHJlY29yZHMuZXhpc3RzKCk6CiAgICAgICAgcmV0dXJuIDAKICAgIG1hbmlmZXN0ID0gZm9sZGVyLydydW5fbWFuaWZlc3QuanNvbicKICAgIGlmIG5vdCBtYW5pZmVzdC5pc19maWxlKCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignUmVjb3JkcyB3aXRob3V0IHJ1biBtYW5pZmVzdCcpCiAgICBpZGVudGl0eSA9IGpzb24ubG9hZHMobWFuaWZlc3QucmVhZF90ZXh0KGVuY29kaW5nPSd1dGYtOCcpKQogICAgaWYgaWRlbnRpdHkuZ2V0KCdiM19pbnB1dF9pZGVudGl0eScpICE9IHJ1bl9pZGVudGl0eShyb290KToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdDaGVja3BvaW50IG1ldGhvZC9pbnB1dC9ydW50aW1lIGlkZW50aXR5IG1pc21hdGNoJykKICAgIGxvb2t1cCA9IHtyWydpZCddOiByIGZvciByIGluIHJvd3N9CiAgICBzZWVuID0gc2V0KCkKICAgIGZvciBsaW5lIGluIHJlY29yZHMucmVhZF90ZXh0KGVuY29kaW5nPSd1dGYtOCcpLnNwbGl0bGluZXMoKToKICAgICAgICBpZiBub3QgbGluZS5zdHJpcCgpOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHIgPSBqc29uLmxvYWRzKGxpbmUpCiAgICAgICAgdWlkID0gclsnaWQnXQogICAgICAgIGlmIHVpZCBpbiBzZWVuIG9yIHVpZCBub3QgaW4gbG9va3VwOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdEdXBsaWNhdGUvdW5leHBlY3RlZCBjaGVja3BvaW50IElEJykKICAgICAgICBpdGVtID0gbG9va3VwW3VpZF0KICAgICAgICBleHBlY3RlZCA9IHsnYXJtJzonQjMnLCAnZGF0YXNldCc6c2NvcGVbJ2RhdGFzZXQnXSwgJ3NwbGl0JzpzY29wZVsnc3BsaXQnXSwKICAgICAgICAgICAgJ3NlZWQnOnNjb3BlWydzZWVkJ10sICdtb2RlbF9zaGEyNTYnOnJvb3RbJ21vZGVsX3NoYTI1NiddLAogICAgICAgICAgICAncnVudGltZV9jb250cmFjdF9zaGEyNTYnOnJvb3RbJ3J1bnRpbWVfY29udHJhY3Rfc2hhMjU2J10sCiAgICAgICAgICAgICdwcm9tcHRfc2hhMjU2JzppdGVtWydwcm9tcHRfc2hhMjU2J10sICdsYWJlbCc6aXRlbVsnbGFiZWwnXSwgJ3ZhbGlkJzpUcnVlfQogICAgICAgIGlmIGFueShyLmdldChrKSAhPSB2IGZvciBrLHYgaW4gZXhwZWN0ZWQuaXRlbXMoKSk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0NoZWNrcG9pbnQgcm93IGlkZW50aXR5IG1pc21hdGNoJykKICAgICAgICBzLCBscDAsIGxwMSA9IHJbJ3MnXSwgclsnbG9ncHJvYl8wJ10sIHJbJ2xvZ3Byb2JfMSddCiAgICAgICAgaWYgbm90IGFsbCh0eXBlKHgpIGluIChpbnQsIGZsb2F0KSBhbmQgbWF0aC5pc2Zpbml0ZSh4KSBmb3IgeCBpbiAocyxscDAsbHAxKSkgb3Igbm90IDAgPD0gcyA8PSAxOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdJbnZhbGlkIG51bWVyaWMgY2hlY2twb2ludCBzY29yZScpCiAgICAgICAgZGlmZiA9IGxwMS1scDAKICAgICAgICBleHBfcyA9IDEvKDErbWF0aC5leHAoLWRpZmYpKSBpZiBkaWZmID49IDAgZWxzZSBtYXRoLmV4cChkaWZmKS8oMSttYXRoLmV4cChkaWZmKSkKICAgICAgICBpZiBhYnMocy1leHBfcykgPiAxZS0xMjoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignU2NvcmUvbG9ncHJvYiBtaXNtYXRjaCcpCiAgICAgICAgcmVmID0gY2hlY2tlZF9uYW1lKHJbJ3J1bnRpbWVfcHJvdmVuYW5jZSddKQogICAgICAgIHByb3ZfcGF0aCA9IG91dF9yb290L3JlZgogICAgICAgIGlmIG5vdCBwcm92X3BhdGguaXNfZmlsZSgpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdNaXNzaW5nIHJlZmVyZW5jZWQgcnVudGltZSBwcm92ZW5hbmNlJykKICAgICAgICBwcm92ID0ganNvbi5sb2Fkcyhwcm92X3BhdGgucmVhZF90ZXh0KGVuY29kaW5nPSd1dGYtOCcpKQogICAgICAgIGlmIChwcm92LmdldCgncnVudGltZV9jb250cmFjdF9zaGEyNTYnKSAhPSByb290WydydW50aW1lX2NvbnRyYWN0X3NoYTI1NiddCiAgICAgICAgICAgICAgICBvciBwcm92LmdldCgncnVubmVyX3NoYTI1NicpICE9IHJvb3RbJ2NvZGVfc2hhMjU2J11bJ2thZ2dsZV9pbmZlcmVuY2UucHknXSk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0ludmFsaWQgcHJvdmVuYW5jZSBpZGVudGl0eScpCiAgICAgICAgbG9nID0gcHJvdl9wYXRoLnBhcmVudC8nbW9kZWxfbG9hZF9vZmZsb2FkLmxvZycKICAgICAgICBpZiBub3QgbG9nLmlzX2ZpbGUoKSBvciBzaGEobG9nLnJlYWRfYnl0ZXMoKSkgIT0gcHJvdi5nZXQoJ29mZmxvYWRfbG9nX3NoYTI1NicpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdNaXNzaW5nL2NoYW5nZWQgbmF0aXZlIGV2aWRlbmNlJykKICAgICAgICBzZWVuLmFkZCh1aWQpCiAgICByZXR1cm4gbGVuKHNlZW4pCgoKZGVmIG1ha2VfY2hlY2twb2ludChvdXRfcm9vdCwgdGFyZ2V0LCByb290LCByb3dzKToKICAgIGNoZWNrZWRfcmVjb3JkcyhvdXRfcm9vdCwgcm9vdCwgcm93cykKICAgIG1lbWJlcnMgPSByZWFkX21lbWJlcnMob3V0X3Jvb3QpCiAgICBtZW1iZXJzLnBvcCgnY2hlY2twb2ludF9tYW5pZmVzdC5qc29uJywgTm9uZSkKICAgIG1hbmlmZXN0ID0geydpZGVudGl0eSc6cnVuX2lkZW50aXR5KHJvb3QpLCAnZmlsZXNfc2hhMjU2Jzp7azpzaGEodikgZm9yIGssdiBpbiBtZW1iZXJzLml0ZW1zKCl9fQogICAgbWFuaWZlc3RbJ3NlYWxfc2hhMjU2J10gPSBzZWFsKG1hbmlmZXN0KQogICAgbWVtYmVyc1snY2hlY2twb2ludF9tYW5pZmVzdC5qc29uJ10gPSBlbmNvZGVkKG1hbmlmZXN0KQogICAgd3JpdGVfemlwKHRhcmdldCwgbWVtYmVycykKCgpkZWYgcmVzdG9yZV9jaGVja3BvaW50KHNvdXJjZSwgb3V0X3Jvb3QsIHJvb3QsIHJvd3MpOgogICAgb3V0X3Jvb3QgPSBQYXRoKG91dF9yb290KQogICAgaWYgb3V0X3Jvb3QuZXhpc3RzKCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignUmVzdW1lIGRlc3RpbmF0aW9uIGFscmVhZHkgZXhpc3RzJykKICAgIG1lbWJlcnMgPSByZWFkX21lbWJlcnMoc291cmNlKQogICAgaWYgJ2NoZWNrcG9pbnRfbWFuaWZlc3QuanNvbicgbm90IGluIG1lbWJlcnM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignTWlzc2luZyBjaGVja3BvaW50IG1hbmlmZXN0JykKICAgIG1hbmlmZXN0ID0ganNvbi5sb2FkcyhtZW1iZXJzLnBvcCgnY2hlY2twb2ludF9tYW5pZmVzdC5qc29uJykpCiAgICBpZiBtYW5pZmVzdC5nZXQoJ3NlYWxfc2hhMjU2JykgIT0gc2VhbChtYW5pZmVzdCkgb3IgbWFuaWZlc3RbJ2lkZW50aXR5J10gIT0gcnVuX2lkZW50aXR5KHJvb3QpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0NoZWNrcG9pbnQgc2VhbC9pZGVudGl0eSBtaXNtYXRjaCcpCiAgICB2ZXJpZnlfaW52ZW50b3J5KG1lbWJlcnMsIG1hbmlmZXN0WydmaWxlc19zaGEyNTYnXSkKICAgIG91dF9yb290LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBzdGFnaW5nID0gUGF0aCh0ZW1wZmlsZS5ta2R0ZW1wKHByZWZpeD0ncmVzdG9yZV8nLCBkaXI9b3V0X3Jvb3QucGFyZW50KSkKICAgIHRyeToKICAgICAgICBmb3IgbmFtZSxyYXcgaW4gbWVtYmVycy5pdGVtcygpOgogICAgICAgICAgICBkZXN0ID0gc3RhZ2luZy9jaGVja2VkX25hbWUobmFtZSkKICAgICAgICAgICAgZGVzdC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgICAgICAgICBkZXN0LndyaXRlX2J5dGVzKHJhdykKICAgICAgICBjaGVja2VkX3JlY29yZHMoc3RhZ2luZywgcm9vdCwgcm93cykKICAgICAgICBvcy5yZXBsYWNlKHN0YWdpbmcsIG91dF9yb290KQogICAgZmluYWxseToKICAgICAgICBpZiBzdGFnaW5nLmV4aXN0cygpOgogICAgICAgICAgICBzaHV0aWwucm10cmVlKHN0YWdpbmcpCgoKZGVmIG5hdGl2ZV9sb2dfY2hlY2sodGV4dCwgY29udHJhY3QpOgogICAgcmVxdWlyZWQgPSBjb250cmFjdFsnZ3B1J11bJ3JlcXVpcmVkX29mZmxvYWRlZF9sYXllcnMnXQogICAgcGFpcnMgPSByZS5maW5kYWxsKHInb2ZmbG9hZGVkXHMrKFxkKylccyovXHMqKFxkKylccytsYXllcnM/XHMrdG9ccytHUFUnLCB0ZXh0LCByZS5JKQogICAgaWYgbm90IGFueShpbnQoYSkgPT0gaW50KGIpID09IHJlcXVpcmVkIGZvciBhLGIgaW4gcGFpcnMpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0Z1bGwgR1BVIG9mZmxvYWQgbm90IHZlcmlmaWVkJykKICAgIGZvciBmaWVsZCBpbiAoJ25fY3R4Jywnbl9iYXRjaCcsJ25fdWJhdGNoJyk6CiAgICAgICAgdmFsdWVzID0gcmUuZmluZGFsbChyJ1xiJytmaWVsZCtyJ1xzKj1ccyooXGQrKVxiJywgdGV4dCkKICAgICAgICBpZiBub3QgdmFsdWVzIG9yIGludCh2YWx1ZXNbLTFdKSAhPSBjb250cmFjdFsnY29uc3RydWN0b3InXVtmaWVsZF06CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0VmZmVjdGl2ZSAnK2ZpZWxkKycgZGlmZmVycyBmcm9tIGNvbnRyYWN0JykKICAgIGRldmljZXMgPSBzZXQocmUuZmluZGFsbChyJ3VzaW5nIGRldmljZSBDVURBKFxkKylccypcKFteXG4pXSpcYlQ0XGJbXlxuKV0qXCknLCB0ZXh0LCByZS5JKSkKICAgIGRldmljZXMudXBkYXRlKHJlLmZpbmRhbGwocidEZXZpY2VccysoXGQrKTpccytbXlxuXSpcYlQ0XGInLCB0ZXh0KSkKICAgIGlmIGRldmljZXMgIT0gc2V0KG1hcChzdHIsIHJhbmdlKGNvbnRyYWN0WydncHUnXVsnZGV2aWNlX2NvdW50J10pKSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignVHdvIGRpc3RpbmN0IFQ0IGRldmljZXMgbm90IHZlcmlmaWVkJykK')
assert hashlib.sha256(raw).hexdigest() == '730b1a13f045e3504fe7a7fe52c5ff177e6cf07e8af77a8c0ef64d797a89fb90'
Path('/kaggle/working/b3_delivery_runtime.py').write_bytes(raw)
raw = base64.b64decode('aW1wb3J0IGNvbnRleHRsaWIKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGltcG9ydGxpYi5tZXRhZGF0YQppbXBvcnQganNvbgppbXBvcnQgbXVsdGlwcm9jZXNzaW5nCmltcG9ydCBvcwppbXBvcnQgcGxhdGZvcm0KaW1wb3J0IHJlCmltcG9ydCBzaHV0aWwKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCkVYUEVDVEVEX0NPTlRSQUNUX1NIQTI1NiA9ICc0YzljM2JjZmY1ZTlmZDMyNTMyZDdlZGY4NWRmN2E5NjFlNDVlYzljNjRiZWUzZGY4ODMzZDk4NmEyZWIyNzZlJwpDT05UUkFDVF9QQVRIID0gUGF0aCgnL2thZ2dsZS93b3JraW5nL2luZmVyZW5jZV9ydW50aW1lX2xvY2suanNvbicpCgpkZWYgZGlnZXN0KHBhdGgpOgogICAgaCA9IGhhc2hsaWIuc2hhMjU2KCkKICAgIHdpdGggUGF0aChwYXRoKS5vcGVuKCdyYicpIGFzIGY6CiAgICAgICAgZm9yIGJsb2NrIGluIGl0ZXIobGFtYmRhOiBmLnJlYWQoMTAyNCAqIDEwMjQpLCBiJycpOiBoLnVwZGF0ZShibG9jaykKICAgIHJldHVybiBoLmhleGRpZ2VzdCgpCgpkZWYgbG9hZF9jb250cmFjdChwYXRoPUNPTlRSQUNUX1BBVEgpOgogICAgcmF3ID0gUGF0aChwYXRoKS5yZWFkX2J5dGVzKCkKICAgIGlmIGhhc2hsaWIuc2hhMjU2KHJhdykuaGV4ZGlnZXN0KCkgIT0gRVhQRUNURURfQ09OVFJBQ1RfU0hBMjU2OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ1J1bnRpbWUgY29udHJhY3QgZmlsZSBTSEEtMjU2IG1pc21hdGNoJykKICAgIGMgPSBqc29uLmxvYWRzKHJhdy5kZWNvZGUoJ3V0Zi04JykpCiAgICB1bnNpZ25lZCA9IHtrOiB2IGZvciBrLCB2IGluIGMuaXRlbXMoKSBpZiBrICE9ICdpbnRlZ3JpdHknfQogICAgc2VhbCA9IGhhc2hsaWIuc2hhMjU2KGpzb24uZHVtcHModW5zaWduZWQsIHNvcnRfa2V5cz1UcnVlLCBzZXBhcmF0b3JzPSgnLCcsICc6JyksIGVuc3VyZV9hc2NpaT1GYWxzZSkuZW5jb2RlKCd1dGYtOCcpKS5oZXhkaWdlc3QoKQogICAgaWYgc2VhbCAhPSBjWydpbnRlZ3JpdHknXVsnc2hhMjU2J106CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignUnVudGltZSBjb250cmFjdCBjYW5vbmljYWwgaW50ZWdyaXR5IG1pc21hdGNoJykKICAgIHJldHVybiBjCgpkZWYgY29tbWFuZChhcmdzKToKICAgIHAgPSBzdWJwcm9jZXNzLnJ1bihhcmdzLCBjaGVjaz1UcnVlLCB0ZXh0PVRydWUsIGNhcHR1cmVfb3V0cHV0PVRydWUpCiAgICByZXR1cm4gcC5zdGRvdXQgKyBwLnN0ZGVycgoKZGVmIGFzc2VydF9ydW50aW1lKGMpOgogICAgYWN0dWFsID0ge25hbWU6IGltcG9ydGxpYi5tZXRhZGF0YS52ZXJzaW9uKG5hbWUpIGZvciBuYW1lIGluIGNbJ3BhY2thZ2VzJ119CiAgICBpZiBhY3R1YWwgIT0gY1sncGFja2FnZXMnXTogcmFpc2UgUnVudGltZUVycm9yKGYnUGFja2FnZSBtaXNtYXRjaDoge2FjdHVhbH0nKQogICAgaWYgcGxhdGZvcm0uc3lzdGVtKCkgIT0gJ0xpbnV4JyBvciAnLicuam9pbihtYXAoc3RyLCBzeXMudmVyc2lvbl9pbmZvWzoyXSkpICE9IGNbJ2J1aWxkJ11bJ3B5dGhvbl9mYW1pbHknXToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0NvbnRyYWN0IHJlcXVpcmVzIEthZ2dsZSBMaW51eC9QeXRob24gMy4xMycpCiAgICBncHUgPSBjb21tYW5kKFsnbnZpZGlhLXNtaScsICctLXF1ZXJ5LWdwdT1pbmRleCx1dWlkLG5hbWUsbWVtb3J5LmZyZWUsbWVtb3J5LnRvdGFsLGRyaXZlcl92ZXJzaW9uJywgJy0tZm9ybWF0PWNzdixub2hlYWRlciddKQogICAgbGluZXMgPSBbcyBmb3IgcyBpbiBncHUuc3BsaXRsaW5lcygpIGlmIHMuc3RyaXAoKV0KICAgIGlmIGxlbihsaW5lcykgIT0gY1snZ3B1J11bJ2RldmljZV9jb3VudCddIG9yIGFueSgnVDQnIG5vdCBpbiBzIGZvciBzIGluIGxpbmVzKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZidDb250cmFjdCByZXF1aXJlcyAyIFQ0IEdQVXM6IHtncHV9JykKICAgIGRpc3QgPSBpbXBvcnRsaWIubWV0YWRhdGEuZGlzdHJpYnV0aW9uKCdsbGFtYS1jcHAtcHl0aG9uJykKICAgIG9yaWdpbiA9IGpzb24ubG9hZHMoZGlzdC5yZWFkX3RleHQoJ2RpcmVjdF91cmwuanNvbicpIG9yICd7fScpCiAgICBpZiBvcmlnaW4uZ2V0KCdhcmNoaXZlX2luZm8nLCB7fSkuZ2V0KCdoYXNoZXMnLCB7fSkuZ2V0KCdzaGEyNTYnKSAhPSBjWydiYWNrZW5kX3NvdXJjZSddWydzZGlzdF9zaGEyNTYnXToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0JhY2tlbmQgaW5zdGFsbGF0aW9uIGlzIG5vdCBmcm9tIHRoZSB2ZXJpZmllZCBzb3VyY2UgYXJjaGl2ZScpCiAgICByZXR1cm4gYWN0dWFsLCBncHUKCmRlZiBwcm92ZW5hbmNlKGMsIHJ1bm5lciwgdGFyZ2V0KToKICAgIHBhY2thZ2VzLCBncHUgPSBhc3NlcnRfcnVudGltZShjKQogICAgZGlzdCA9IGltcG9ydGxpYi5tZXRhZGF0YS5kaXN0cmlidXRpb24oJ2xsYW1hLWNwcC1weXRob24nKQogICAgYmluYXJpZXMgPSB7fQogICAgZm9yIGVudHJ5IGluIGRpc3QuZmlsZXMgb3IgW106CiAgICAgICAgaWYgc3RyKGVudHJ5KS5lbmRzd2l0aCgoJy5zbycsICcuZGxsJywgJy5keWxpYicpKToKICAgICAgICAgICAgcCA9IFBhdGgoZGlzdC5sb2NhdGVfZmlsZShlbnRyeSkpOyBiaW5hcmllc1tzdHIoZW50cnkpXSA9IGRpZ2VzdChwKQogICAgaWYgbm90IGJpbmFyaWVzOiByYWlzZSBSdW50aW1lRXJyb3IoJ05vIGNvbXBpbGVkIGJhY2tlbmQgbGlicmFyeSBmb3VuZCBmb3IgcHJvdmVuYW5jZScpCiAgICByZXN1bHQgPSBkaWN0KHJ1bnRpbWVfY29udHJhY3Rfc2hhMjU2PUVYUEVDVEVEX0NPTlRSQUNUX1NIQTI1NiwKICAgICAgICBydW50aW1lX2NvbnRyYWN0X2ludGVncml0eT1jWydpbnRlZ3JpdHknXVsnc2hhMjU2J10sIHBhY2thZ2VzPXBhY2thZ2VzLAogICAgICAgIHB5dGhvbj1zeXMudmVyc2lvbiwgcGxhdGZvcm09cGxhdGZvcm0ucGxhdGZvcm0oKSwgbGliYz1wbGF0Zm9ybS5saWJjX3ZlcigpLAogICAgICAgIGdwdV9iZWZvcmVfbG9hZD1ncHUsIGNwdV9jb3VudD1tdWx0aXByb2Nlc3NpbmcuY3B1X2NvdW50KCksCiAgICAgICAgbl90aHJlYWRzPW1heChtdWx0aXByb2Nlc3NpbmcuY3B1X2NvdW50KCkgLy8gMiwgMSksIG5fdGhyZWFkc19iYXRjaD1tdWx0aXByb2Nlc3NpbmcuY3B1X2NvdW50KCksCiAgICAgICAgY29uc3RydWN0b3I9Y1snY29uc3RydWN0b3InXSwgcnVubmVyX3NoYTI1Nj1kaWdlc3QocnVubmVyKSwKICAgICAgICBjdWRhX2RldmljZV9vcmRlcj1vcy5lbnZpcm9uLmdldCgnQ1VEQV9ERVZJQ0VfT1JERVInKSwKICAgICAgICBjdWRhX3Zpc2libGVfZGV2aWNlcz1vcy5lbnZpcm9uLmdldCgnQ1VEQV9WSVNJQkxFX0RFVklDRVMnKSwKICAgICAgICBiYWNrZW5kX3NkaXN0X3NoYTI1Nj1jWydiYWNrZW5kX3NvdXJjZSddWydzZGlzdF9zaGEyNTYnXSwgY29tcGlsZWRfYmFja2VuZF9zaGEyNTY9YmluYXJpZXMsCiAgICAgICAgYnVpbGRfZW52aXJvbm1lbnQ9e2s6IG9zLmVudmlyb24uZ2V0KGspIGZvciBrIGluICgnQ01BS0VfQVJHUycsICdDTUFLRV9CVUlMRF9QQVJBTExFTF9MRVZFTCcsICdGT1JDRV9DTUFLRScpfSkKICAgIGZvciBuYW1lLCBhcmdzIGluIFsoJ252Y2MnLCBbJ252Y2MnLCAnLS12ZXJzaW9uJ10pLCAoJ2NvbXBpbGVyJywgWydjYycsICctLXZlcnNpb24nXSksCiAgICAgICAgICAgICAgICAgICAgICAoJ2N4eF9jb21waWxlcicsIFsnYysrJywgJy0tdmVyc2lvbiddKSwgKCdjbWFrZScsIFsnY21ha2UnLCAnLS12ZXJzaW9uJ10pLAogICAgICAgICAgICAgICAgICAgICAgKCdwaXBfZnJlZXplJywgW3N5cy5leGVjdXRhYmxlLCAnLW0nLCAncGlwJywgJ2ZyZWV6ZScsICctLWFsbCddKV06CiAgICAgICAgcmVzdWx0W25hbWVdID0gY29tbWFuZChhcmdzKQogICAgdGFyZ2V0ID0gUGF0aCh0YXJnZXQpOyB0YXJnZXQucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHRhcmdldC53cml0ZV90ZXh0KGpzb24uZHVtcHMocmVzdWx0LCBpbmRlbnQ9MiksIGVuY29kaW5nPSd1dGYtOCcpCiAgICByZXR1cm4gcmVzdWx0CgpkZWYgbG9hZF9sb2NrZWRfbW9kZWwoYywgbW9kZWxfcGF0aCwgdG9rZW5pemVyX21hbmlmZXN0LCBncHVfbGF5ZXJzLCBsb2dfcGF0aCwgbW9kZWxfZGlnZXN0KToKICAgIGltcG9ydCBsbGFtYV9jcHAKICAgIGlmIG1vZGVsX2RpZ2VzdCAhPSBjWydtb2RlbCddWydzaGEyNTYnXTogcmFpc2UgVmFsdWVFcnJvcignRnJvemVuIG1vZGVsIFNIQS0yNTYgbWlzbWF0Y2gnKQogICAgaWYgZ3B1X2xheWVycyAhPSBjWydjb25zdHJ1Y3RvciddWyduX2dwdV9sYXllcnMnXTogcmFpc2UgVmFsdWVFcnJvcignR1BVIGxheWVycyBkaWZmZXIgZnJvbSBsb2NrJykKICAgIGlmICh0b2tlbml6ZXJfbWFuaWZlc3RbJ3Rva2VuaXplciddLCB0b2tlbml6ZXJfbWFuaWZlc3RbJ3Rva2VuaXplcl9yZXZpc2lvbiddLCB0b2tlbml6ZXJfbWFuaWZlc3RbJ3Byb21wdF9idWRnZXRfdG9rZW5zJ10pICE9ICgKICAgICAgICBjWyd0b2tlbml6ZXInXVsncmVwbyddLCBjWyd0b2tlbml6ZXInXVsncmV2aXNpb24nXSwgY1snY29uc3RydWN0b3InXVsnbl9jdHgnXSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignVG9rZW5pemVyL2NvbnRleHQgZGlmZmVyIGZyb20gcnVudGltZSBsb2NrJykKICAgIGlmIG5vdCBsbGFtYV9jcHAubGxhbWFfc3VwcG9ydHNfZ3B1X29mZmxvYWQoKTogcmFpc2UgUnVudGltZUVycm9yKCdDVURBIGJhY2tlbmQgdW5hdmFpbGFibGUnKQogICAga3dhcmdzID0gZGljdChjWydjb25zdHJ1Y3RvciddKQogICAga3dhcmdzWydzcGxpdF9tb2RlJ10gPSBnZXRhdHRyKGxsYW1hX2NwcCwga3dhcmdzWydzcGxpdF9tb2RlJ10pCiAgICBrd2FyZ3MudXBkYXRlKG5fdGhyZWFkcz1tYXgobXVsdGlwcm9jZXNzaW5nLmNwdV9jb3VudCgpIC8vIDIsIDEpLCBuX3RocmVhZHNfYmF0Y2g9bXVsdGlwcm9jZXNzaW5nLmNwdV9jb3VudCgpKQogICAgcGF0aCA9IFBhdGgobG9nX3BhdGgpCiAgICB3aXRoIHBhdGgub3BlbigndycsIGVuY29kaW5nPSd1dGYtOCcpIGFzIGxvZywgY29udGV4dGxpYi5yZWRpcmVjdF9zdGRlcnIobG9nKToKICAgICAgICBtb2RlbCA9IGxsYW1hX2NwcC5MbGFtYShtb2RlbF9wYXRoPXN0cihtb2RlbF9wYXRoKSwgdmVyYm9zZT1UcnVlLCAqKmt3YXJncykKICAgIHRleHQgPSBwYXRoLnJlYWRfdGV4dChlbmNvZGluZz0ndXRmLTgnLCBlcnJvcnM9J3JlcGxhY2UnKQogICAgcHJpbnQodGV4dCwgZmx1c2g9VHJ1ZSkKICAgIGZyb20gYjNfZGVsaXZlcnlfcnVudGltZSBpbXBvcnQgbmF0aXZlX2xvZ19jaGVjawogICAgdHJ5OgogICAgICAgIG5hdGl2ZV9sb2dfY2hlY2sodGV4dCwgYykKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgbW9kZWwuY2xvc2UoKQogICAgICAgIHJhaXNlCiAgICByZXR1cm4gbW9kZWwK')
assert hashlib.sha256(raw).hexdigest() == '27b57d69089d009b225e3653dd38afed14bb48442765652c2c80c0c38df01775'
Path('/kaggle/working/grace_runtime_contract.py').write_bytes(raw)
raw = base64.b64decode('...')
assert hashlib.sha256(raw).hexdigest() == '059c5fca1de0043f5b7710ed39c79976301d8b92548e3f2f85aaaf2afeb08fe8'
Path('/kaggle/working/kaggle_inference.py').write_bytes(raw)
raw = base64.b64decode('...')
assert hashlib.sha256(raw).hexdigest() == '4c9c3bcff5e9fd32532d7edf85df7a961e45ec9c64bee3df8833d986a2eb276e'
Path('/kaggle/working/inference_runtime_lock.json').write_bytes(raw)
raw = base64.b64decode('ewogICJhdXRob3JpdHkiOiB7CiAgICAibmV3X2h1bWFuX3Jldmlld19jbGFpbWVkIjogZmFsc2UsCiAgICAibmV3X3Njb3BlX3JhdGlmaWNhdGlvbl90aW1lc3RhbXBfY2xhaW1lZCI6IGZhbHNlLAogICAgInNjb3BlIjogIkV4aXN0aW5nIHJldmlld2VkIERldmlnbiBDYXNlMDg7IG9wZXJhdGlvbmFsIGZpeGVzIHJlcXVlc3RlZCBieSB1c2VyIgogIH0sCiAgImJ1bmRsZV9maWxlbmFtZSI6ICJiM19kZXZpZ25fdmFsaWRhdGlvbl92Ni56aXAiLAogICJidW5kbGVfc2hhMjU2IjogIjMwZDI0NmRkZGQyMDVjOTU5YmRhMGNjY2Q4M2JkMjBlNWQwYTkwYzMwYjc1MTkwODMxYWFjMzY1NjlhYWYwNmIiLAogICJjb2RlX3NoYTI1NiI6IHsKICAgICJiM19kZWxpdmVyeV9ydW50aW1lLnB5IjogIjczMGIxYTEzZjA0NWUzNTA0ZmU3YTdmZTUyYzVmZjE3N2U2Y2YwN2U4YWY3N2E4YzBlZjY0ZDc5N2E4OWZiOTAiLAogICAgImdyYWNlX3J1bnRpbWVfY29udHJhY3QucHkiOiAiMjdiNTdkNjkwODlkMDA5YjIyNWUzNjUzZGQzOGFmZWQxNGJiNDg0NDI3NjU2NTJjMmM4MGMwYzM4ZGYwMTc3NSIsCiAgICAiaW5mZXJlbmNlX3J1bnRpbWVfbG9jay5qc29uIjogIjRjOWMzYmNmZjVlOWZkMzI1MzJkN2VkZjg1ZGY3YTk2MWU0NWVjOWM2NGJlZTNkZjg4MzNkOTg2YTJlYjI3NmUiLAogICAgImthZ2dsZV9pbmZlcmVuY2UucHkiOiAiMDU5YzVmY2ExZGUwMDQzZjViNzcxMGVkMzljNzk5NzYzMDFkOGI5MjU0OGUzZjJmODVhYWFmMmFmZWIwOGZlOCIKICB9LAogICJjcmVhdGVkX2F0X3V0YyI6ICIyMDI2LTExLTA4VDE1OjMxOjE5LjQ3NTYxNiswMDowMCIsCiAgImRpcnR5X3NvdXJjZV9pbnZlbnRvcnkiOiB7CiAgICAic2NyaXB0c1xcYjNfZGVsaXZlcnlfcnVudGltZS5weSI6ICI3MzBiMWExM2YwNDVlMzUwNGZlN2E3ZmU1MmM1ZmYxNzdlNmNmMDdlOGFmNzdhOGMwZWY2NGQ3OTdhODlmYjkwIiwKICAgICJzY3JpcHRzXFxidWlsZF9iM19kZWxpdmVyeV92Ni5weSI6ICJlOTg2ZTQxZGNlYTU5YTY4MGIzZjU2OTcxYzI5NmQ2YWRhMzIwMjA5MWZiNWZiYjAwZTI0NDQ0MDMzNTcxNTk2IgogIH0sCiAgImV4Y2x1ZGVkIjogewogICAgInJldmVhbCI6ICJCTE9DS0VEX0VNUFRZX1ZFUklGSUVEX1BPT0wiLAogICAgInRlc3QiOiAiQkxPQ0tFRF9WQUxJREFUSU9OX1RIUkVTSE9MRCIKICB9LAogICJnaXRfY29tbWl0IjogIjY4YjdiZjEzN2JiOWI5NTcwNTFjNDZhODZkNTNlYzQxYTUyODczMWMiLAogICJpbmZlcmVuY2Vfcm93c19zaGEyNTYiOiAiNWQ4ODk2MmFiMmNiYTlmMzY0NjE2YjQ3MTgxZTQ1MmE2MjhhYTI2ZWMyZGU4ZTBhNGIzYzAyMGY1ZGIyMjIwZCIsCiAgImxhYmVsc19zb3VyY2Vfc2hhMjU2IjogImMyNjg1NDc2YmZmZmIwYzAyZDk5ZjlhYTQ0M2E0ZGI0OGNjM2ViZDE5ODdiNGY3MWVkYjQ4OTAyMjNhYzc5YTUiLAogICJtZW1iZXJzX3NoYTI1NiI6IHsKICAgICJhdWRpdGVkX2NlbnN1cy5qc29ubCI6ICJkZWU0YTMzMDA5ZjRlY2JmNDJiOTZlNmYxOGM1ZjQzOTBkNWYwYTYyMDM3MGFmOTI1YmFlMGYyY2NlZWI3ZTU5IiwKICAgICJidW5kbGVfbWFuaWZlc3QuanNvbiI6ICIyYzliYzU3N2FmNWRmNWRkN2I2OGY1YjMxNTdjNTVjZWI4N2Y2MGFjODU5YzM5NDY3ZDMzNDUyYzE2MzA2NjI1IiwKICAgICJldmlkZW5jZS9lbGlnaWJsZV9wYWlycy5kZXZpZ24uanNvbmwiOiAiMzA2YjNjMzM5MWIyMTY0OTYzNWQxNzU3YWYzNjVmYzlkZTVmYjkyODg5NDk4OWJmZTM4MmZkNjQ5OGE5ZjhmOSIsCiAgICAiZXZpZGVuY2UvcGFyZW50X2lucHV0c19tYW5pZmVzdC5qc29uIjogImUwZmU2YjFkZTI4NzY1YWViMzg4MzE2ZGRkNzhiNmRlZjc4NmRhNDE5OTRjNmE0ZDA0OWNjNzAxMjNiOTJkMTMiLAogICAgImV2aWRlbmNlL3BhcmVudF9wb29sX21hbmlmZXN0Lmpzb24iOiAiZmI4NTZjZWM2YzIyZDExYzc5OTA4ZDJlNDJlYzIzNzFjMGUxNzljODllMGRmYTYwOTJjOWY4MDI2ODM1MTk5NCIsCiAgICAiZXZpZGVuY2UvcGFyZW50X3Byb21wdF9tYW5pZmVzdC5qc29uIjogIjZkMzg2M2RhNmIwMGRhNGY5Mjc0ZDIzOWNiOWVhZTE5Y2Y4YzEzMWYzZDVhOTM5ZmUyNDFlNDk1YjFkMjRjOGEiLAogICAgImV2aWRlbmNlL3Jldmlld19kdW5nLmpzb24iOiAiZWRjZmE0MDA4Y2UwY2UyNTU4NWY5MzE0ODhiNTIyOTUyZjNhMzk4NDE3YWY4YjhjYjE5YTZmYWE4MjEzMWRmNyIsCiAgICAiZXZpZGVuY2UvcmV2aWV3X2hpZXUuanNvbiI6ICIwY2M5N2MyMmVmODBiOTY2NzMyZDNkZjFhMzUyMWIxOTY0NWE5OTZhNWU4YjFjOTgxODAwMThjYmQ0M2VmMTIxIiwKICAgICJsYWJlbHMuanNvbiI6ICI5NDE0MmNmOGY0NjY4MmE5MTZkZTdjZTgxNmNjMTRmYzI5ZTE0NTI2OTczZDFiZmI3ZmQ2MDFlYmRjNmEzYjM0IiwKICAgICJwcm9tcHRzLmpzb25sIjogIjVkODg5NjJhYjJjYmE5ZjM2NDYxNmI0NzE4MWU0NTJhNjI4YWEyNmVjMmRlOGUwYTRiM2MwMjBmNWRiMjIyMGQiCiAgfSwKICAibWV0aG9kX2lkIjogImIzLXZlcmlmaWVkLXBhdGNoLXYxIiwKICAibW9kZWxfc2hhMjU2IjogImY2MDUwYjRkYTIyODY5YWEzZmI5ZDk0YmIzMzE0ZTlmYzVhMDI3ZDQ3NDFjMWY5Nzc2MWMzODdkNmIyYjIzMzUiLAogICJwYXJlbnRfcmVsZWFzZSI6ICJyZWxlYXNlXzIwMjYxMTA4X3Y1IiwKICAicG9vbF9zaGEyNTYiOiAiMzA2YjNjMzM5MWIyMTY0OTYzNWQxNzU3YWYzNjVmYzlkZTVmYjkyODg5NDk4OWJmZTM4MmZkNjQ5OGE5ZjhmOSIsCiAgInByb21wdF9jb250ZW50X3NoYTI1NiI6ICJkZWU0YTMzMDA5ZjRlY2JmNDJiOTZlNmYxOGM1ZjQzOTBkNWYwYTYyMDM3MGFmOTI1YmFlMGYyY2NlZWI3ZTU5IiwKICAicmVsZWFzZV9pZCI6ICJyZWxlYXNlXzIwMjYxMTA4X3Y2IiwKICAicmVsZWFzZV9zdGF0dXMiOiAiQ1BVX1BSRVBBUkVEX05BVElWRV9QRU5ESU5HIiwKICAicnVudGltZV9jb250cmFjdF9zaGEyNTYiOiAiNGM5YzNiY2ZmNWU5ZmQzMjUzMmQ3ZWRmODVkZjdhOTYxZTQ1ZWM5YzY0YmVlM2RmODgzM2Q5ODZhMmViMjc2ZSIsCiAgInNjaGVtYV92ZXJzaW9uIjogMSwKICAic2NvcGUiOiB7CiAgICAiYXJtIjogIkIzIiwKICAgICJjb3VudCI6IDI3MzIsCiAgICAiZGF0YXNldCI6ICJkZXZpZ24iLAogICAgInNlZWQiOiA0MiwKICAgICJzcGxpdCI6ICJ2YWxpZGF0aW9uIgogIH0sCiAgInNjb3JpbmciOiAidW5jb25zdHJhaW5lZF9maXJzdF90b2tlbl9sb2dpdHNfdjEiLAogICJzZWFsX3NoYTI1NiI6ICJmZjQ0YjliZjQzYjc5NGJmZjA1NmUzNmI4MDI0NmM2NDFlNWVmOTVjZTA4YTgxOWQwMmMwYmU1NjEzYWJkMzA4IiwKICAidG9rZW5pemVyX3JldmlzaW9uIjogImIyY2ZmNjQ2ZWI0YmIxZDY4MzU1YzAxYjE4YWUwMmU3Y2Y0MmQxMjAiCn0K')
assert hashlib.sha256(raw).hexdigest() == '38ed52ee4cc71ad41c1f5c94af41afa4e4c84a842cf2b0daa46e5cb1afe6fc36'
Path('/kaggle/working/inference_input_manifest.json').write_bytes(raw)
os.environ['B3_INPUT_ROOT_SHA256'] = '38ed52ee4cc71ad41c1f5c94af41afa4e4c84a842cf2b0daa46e5cb1afe6fc36'
sys.path.insert(0, '/kaggle/working')
from b3_delivery_runtime import (load_input_root, discover_bundle, validate_bundle,
    checked_records, restore_checkpoint, make_checkpoint, write_zip)
from grace_runtime_contract import assert_runtime, digest, load_contract, EXPECTED_CONTRACT_SHA256
RUNTIME_LOCK_PATH = Path('/kaggle/working/inference_runtime_lock.json')
RUNTIME = load_contract()
INPUT_ROOT = load_input_root('/kaggle/working/inference_input_manifest.json', os.environ['B3_INPUT_ROOT_SHA256'])
note('Independent input root verified: '+os.environ['B3_INPUT_ROOT_SHA256'])


In [ ]:
ARM = 'B3'
DATASET = 'devign'
SPLIT = 'validation'
SEED = 42
EXPECTED_COUNT = 2732

KAGGLE_INPUT_ROOT = Path('/kaggle/input')
BUNDLE_WORK = Path('/kaggle/working/b3_bundles')
OUT_ROOT = Path('/kaggle/working/b3_devign_validation_runtime_locked')
BUNDLE_NAME = 'b3_devign_validation_v6.zip'
CHECKPOINT_NAME = 'grace_b3_devign_val_checkpoint.zip'
CHECKPOINT_INPUT_PATH = None
RESUME_REQUIRED = False

MODEL_FILENAME = 'Qwen3-Coder-30B-A3B-Instruct-Q4_K_M.gguf'
MODEL_SHA256 = 'f6050b4da22869aa3fb9d94bb3314e9fc5a027d4741c1f97761c387d6b2b2335'
MAX_RUNTIME_HOURS = 9
CHECKPOINT_RESERVE_SECONDS = 300
GPU_LAYERS = -1

# STRICT METHODOLOGICAL GUARD: Validation determines tau* post-inference.
# DO NOT apply or read historical test thresholds (e.g. LOCKED_THRESHOLDS or tau*=0.77).
HISTORICAL_THRESHOLDS_APPLIED = None

if (MODEL_FILENAME, MODEL_SHA256, GPU_LAYERS) != (
    RUNTIME['model']['filename'], RUNTIME['model']['sha256'], RUNTIME['constructor']['n_gpu_layers']
):
    raise ValueError('Notebook model or GPU settings differ from runtime lock')

note(f'Configured scope: ARM={ARM}, DATASET={DATASET}, SPLIT={SPLIT}, SEED={SEED}, COUNT={EXPECTED_COUNT}')


In [ ]:
source = discover_bundle(KAGGLE_INPUT_ROOT, INPUT_ROOT)
bundle_manifest, prompts_records, members = validate_bundle(source, INPUT_ROOT)
BUNDLE_WORK.mkdir(parents=True, exist_ok=True)
bundle_file = BUNDLE_WORK / BUNDLE_NAME
write_zip(bundle_file, members)
# Prepared ZIP must reproduce the independently pinned archive in either input mode.
if digest(bundle_file) != INPUT_ROOT['bundle_sha256']:
    raise ValueError('Prepared bundle identity differs from pin')
if CHECKPOINT_INPUT_PATH is not None:
    restore_checkpoint(Path(CHECKPOINT_INPUT_PATH), OUT_ROOT, INPUT_ROOT, prompts_records)
elif RESUME_REQUIRED:
    raise ValueError('Resume requested without explicit checkpoint path')
elif OUT_ROOT.exists():
    raise ValueError('Fresh destination already exists; set explicit checkpoint path to resume')
else:
    OUT_ROOT.mkdir(parents=True)
records_done = checked_records(OUT_ROOT, INPUT_ROOT, prompts_records)
note(f'Input verified: {len(prompts_records)} prompts, {records_done} existing records')


In [ ]:
import urllib.request
build = RUNTIME['build']
os.environ['CMAKE_ARGS'] = (
    f"-DGGML_CUDA={'on' if build['GGML_CUDA'] else 'off'}"
    f" -DCMAKE_CUDA_ARCHITECTURES={build['CMAKE_CUDA_ARCHITECTURES']}"
    f" -DCMAKE_LIBRARY_PATH={build['CMAKE_LIBRARY_PATH']}"
)
for key in ('CMAKE_BUILD_PARALLEL_LEVEL', 'FORCE_CMAKE'):
    os.environ[key] = build[key]

packages = [f'{name}=={version}' for name, version in RUNTIME['packages'].items()
            if name != RUNTIME['backend_source']['name']]
packages += ['typing-extensions>=4.5.0', 'diskcache>=5.6.1', 'jinja2>=2.11.3']
run_logged([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', *packages], 'install_dependencies')

source = RUNTIME['backend_source']
with urllib.request.urlopen(f"https://pypi.org/pypi/{source['name']}/{source['version']}/json", timeout=60) as response:
    metadata = json.load(response)
artifact = next(item for item in metadata['urls'] if item['filename'] == source['sdist_filename'])
if artifact['digests']['sha256'] != source['sdist_sha256']:
    raise ValueError('Backend source registry SHA-256 mismatch')
archive_path = Path('/kaggle/working') / source['sdist_filename']
with urllib.request.urlopen(artifact['url'], timeout=120) as response, archive_path.open('wb') as target:
    shutil.copyfileobj(response, target)
if digest(archive_path) != source['sdist_sha256']:
    raise ValueError('Backend source file SHA-256 mismatch')
run_logged([sys.executable, '-m', 'pip', 'install', '-v', '--no-cache-dir', '--no-deps',
            '--force-reinstall', str(archive_path)], 'install_llama_cuda')
note('Locked packages: ' + json.dumps(assert_runtime(RUNTIME)[0], sort_keys=True))
diagnostics_zip()


In [ ]:
assert_runtime(RUNTIME)
import llama_cpp
versions()
note('llama-cpp-python GPU offload: ' + str(llama_cpp.llama_supports_gpu_offload()))
assert llama_cpp.llama_supports_gpu_offload(), 'llama-cpp-python CUDA backend unavailable'
diagnostics_zip()


In [ ]:
note('Worker and helper bytes were pinned during bootstrap; no replacement cell is used.')


In [ ]:
deadline = SESSION_STARTED + MAX_RUNTIME_HOURS * 3600
status = OUT_ROOT / f'seed_{SEED}_status.json'
models = sorted(KAGGLE_INPUT_ROOT.rglob(MODEL_FILENAME))
if len(models) != 1:
    raise ValueError('Need exactly one locked model input')
state = 'error'
try:
    if records_done < EXPECTED_COUNT:
        run_logged([sys.executable, '-u', '/kaggle/working/kaggle_inference.py',
            '--model', str(models[0]), '--model-revision', 'local-input',
            '--seed', str(SEED), '--arm', ARM, '--expected-model-sha256', MODEL_SHA256,
            '--gpu-layers', str(GPU_LAYERS), '--deadline-epoch', str(deadline),
            '--checkpoint-reserve-seconds', str(CHECKPOINT_RESERVE_SECONDS), '--status', str(status),
            '--run', DATASET, SPLIT, str(bundle_file), str(OUT_ROOT / f'seed_{SEED}' / DATASET / SPLIT)],
            f'inference_seed_{SEED}')
    records_done = checked_records(OUT_ROOT, INPUT_ROOT, prompts_records)
    state = 'complete' if records_done == EXPECTED_COUNT else 'checkpointed'
finally:
    # Always try to preserve a valid checkpoint, including an inference exception.
    make_checkpoint(OUT_ROOT, Path('/kaggle/working')/CHECKPOINT_NAME, INPUT_ROOT, prompts_records)
    (OUT_ROOT/'batch_summary.json').write_text(json.dumps({'state':state,
        'input_root_sha256':os.environ['B3_INPUT_ROOT_SHA256'], 'historical_threshold_applied':False}),encoding='utf-8')
    diagnostics_zip()
